In [ ]:
import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../experiments'));

import experiment_helper
from periodic_simulation_setup import *
import json
import os

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

In [ ]:

allowBending = False
useTFT = True


name = 'zigzag_line_10_to_90'
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
time_stamp = '2023_11_01_16_27'
base_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(base_folder):
    os.makedirs(base_folder)  

# pressure = 0.8
stiffness_pressure = 0.3
scale_factor_pressure = 0.01
disableFusedRegionTFT = False

In [ ]:
label = 33

In [ ]:
result_folder = "{}/{}".format(base_folder, label)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  


with open('../data/ZigZag_10_90_deg/ZigZag_10_90_deg{}.json'.format(label), 'r') as f:
    data = json.load(f)
fusedVertices = data['FusedVertices']
fusedVertices = [True if vx == 1 else False for vx in fusedVertices]
V = data['Vertices']
V = np.array(V) * 5
F = data['Faces']
m = MeshFEM.Mesh(V, F)
ipu = inflation.InflatablePeriodicUnit(m, fusedVertices)

finalMarkers = np.where(np.array(fusedVertices) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
# result_folder = "{}/{}".format(base_folder, label)
# if not os.path.exists(result_folder):
#     os.makedirs(result_folder)  


# with open('ZigZag_90_deg_EdgeSize_1.50.json') as f:
#     data = json.load(f)
# fusedVertices = data['FusedVertices']
# fusedVertices = [True if vx == 1 else False for vx in fusedVertices]
# V = data['Vertices']
# F = data['Faces']
# m = MeshFEM.Mesh(V, F)
# ipu = inflation.InflatablePeriodicUnit(m, fusedVertices)

# finalMarkers = np.where(np.array(fusedVertices) == 1)[0]
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
# # m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# # m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

# fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
# ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
# import periodic_unit_helper

# h = 5
# w = 5
# avg_len = 1


# # ipu, m, marker = periodic_unit_helper.get_parallel_tube_periodic(h, w, wall_w = 0.1, avg_len = 0.1)

# # finalMarkers = np.where(np.array(marker) == 1)[0]

# # fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
# # ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)


In [ ]:
configure_solver_parallelism()


viewer = TriMeshViewer(ipu, width=1024, height=1024)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
import experiment_helper, importlib
importlib.reload(experiment_helper)

In [ ]:
cr = experiment_helper.helper_run_equilibrium(ipu, allow_bending = False, stiffness_pressure = stiffness_pressure, cb = cb)

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
ipu.gradient(energyType = inflation.InflatablePeriodicUnit.EnergyType.Elastic)

In [ ]:
ipu.gradient(energyType = inflation.InflatablePeriodicUnit.EnergyType.Pressure)

In [ ]:
ipu.gradient()

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(ipu.hessian(), reflect = True)

In [ ]:
from scipy.sparse.linalg import eigs

In [ ]:
# scipy.linalg.eigh(H, eigvals_only = True, subset_by_index = [0, 3])

In [ ]:
stiffness_values, sampled_alphas, stiffness_coefficient = experiment_helper.helper_compute_bending_stiffness(ipu, cb, result_folder = result_folder, name = name, variable = label, render_images = True)

In [ ]:
# import mode_viewer

# import compute_vibrational_modes
# class ModalAnalysisWrapper:
#     def __init__(self, sheet):
#         self.sheet = sheet
#     def hessian(self):
#         return self.sheet.hessian()

# lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(ipu), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=3, sigma=-1e-10, fixedVars = stiffness_fixedVars)


# import mode_viewer, importlib
# mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=20, wireframe = False)
# mview.show()

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
igl.bounding_box(ipu.getVars()[3:-2].reshape(-1, 3))

In [ ]:
np.max(stiffness_values)

In [ ]:
ipu.getVars()

In [ ]:
"{}/stiffness_{}_{}.png".format(result_folder, name, label)

In [ ]:
from IPython.display import Image
Image(filename="{}/stiffness_{}_{}.png".format(result_folder, name, label)) 

In [ ]:
viewer.setCameraParams(((0.05388981383962801, -3.1668002724484476, 5.803170717118252),
 (-0.014947641421525739, 0.8776479009146649, 0.47907278156457567),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.update()
viewer.showWireframe(False)

In [ ]:
orender = viewer.offscreenRenderer(width=1024,height=1024)
# orender.meshes[0].setColor(C[mm.elements().ravel()])
orender.render()
orender.save("zigzag.png")

In [ ]:
# import mode_viewer

# import compute_vibrational_modes
# class ModalAnalysisWrapper:
#     def __init__(self, sheet):
#         self.sheet = sheet
#     def hessian(self):
#         return self.sheet.hessian()

# lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(ipu), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=3, sigma=-1e-10, fixedVars = stiffness_fixedVars)


# import mode_viewer, importlib
# mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=20, wireframe = False)
# mview.show()

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
betas = np.linspace(0, 2 * np.pi, 1000)

In [ ]:
optimizer = inflation.get_inflation_optimizer(ipu, ipu.getStretchingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
stretchingStiffness = inflation.getStretchingStiffness(ipu, betas, optimizer, 0, ipu.getStretchingStiffnessFixedVars())

In [ ]:
plot_min_r = 0
plot_max_r = None
r = list(stretchingStiffness)
# + list(stretchingStiffness)
# theta = list(sampled_alpha) + list(np.pi + np.array(sampled_alpha))
theta = list(betas)

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot(theta, r)
ax.set_rmax(max(stretchingStiffness) if plot_max_r is None else plot_max_r)
ax.set_rmin(min(stretchingStiffness) - 0.2 * (max(stretchingStiffness) - min(stretchingStiffness)) if plot_min_r is None else plot_min_r)
# ax.set_rticks([0.5, 1, 1.5, 2])  # Less radial ticks
# ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
ax.grid(True)

ax.set_title("Stretching stiffness", va='bottom')
plt.tight_layout()
plt.savefig("{}/stretching_stiffness_{}_{}.png".format(result_folder, name, 49), dpi = 300) 

In [ ]:
np.min(stretchingStiffness), np.max(stretchingStiffness)

In [ ]:
import periodic_simulation_setup

In [ ]:
points = periodic_simulation_setup.visualize_average_deformation_gradient(ipu, 100, plot_max_r=1, plot_min_r=0, show_figure=True, filename = "{}/average_deformation_gradient_{}_{}.png".format(result_folder, name, 49))